In [ ]:
!pip -q install torchmetrics[image] torch-fidelity

In [ ]:
from pathlib import Path
import sys
import os
import json
import math
import time
import copy
import random
import traceback
from dataclasses import dataclass, asdict
from itertools import product
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from IPython.display import display

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.utils import make_grid
import torchvision.utils as vutils
import matplotlib.pyplot as plt

from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance

from tqdm.auto import tqdm

In [ ]:
SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 1. Paths Projeto

In [ ]:
ROOT = Path("/content/drive/MyDrive/IAG")
DATA_ROOT = ROOT / "archive"
SRC_ROOT = ROOT / "src"
RESULTS_ROOT = ROOT / "results"

MODELS_DIR = RESULTS_ROOT / "models"
IMAGES_DIR = RESULTS_ROOT / "images"
METRICS_DIR = RESULTS_ROOT / "metrics"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

TRAINING_CSV_PATH = DATA_ROOT / "training_20_percent.csv"

print("ROOT:", ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("SRC_ROOT:", SRC_ROOT)
print("TRAINING_CSV_PATH:", TRAINING_CSV_PATH)

# 2. Dataset

## 2.1 Loader (professor)

In [ ]:
sys.path.append(str(SRC_ROOT))
from artbench_local_dataset import load_kaggle_artbench10_splits

## 2.2 Carregar dataset

In [ ]:
hf_ds = load_kaggle_artbench10_splits(DATA_ROOT)
train_hf = hf_ds["train"]
test_hf = hf_ds["test"]

print("Train size:", len(train_hf))
print("Test size :", len(test_hf))
print("Columns   :", train_hf.column_names)

label_feature = train_hf.features["label"]
class_names = list(label_feature.names)
num_classes = len(class_names)
print("Num classes:", num_classes)
print("Class names:", class_names)

## 2.3 Helpers dataset

In [ ]:
def load_ids_from_training_csv(csv_path, index_column="train_id_original"):
    df = pd.read_csv(csv_path)
    if index_column not in df.columns:
        raise ValueError(f"Coluna '{index_column}' não encontrada no CSV.")
    return df[index_column].astype(int).tolist()

def safe_num_workers(requested_workers=2):
    # Keep 0 for Colab stability
    return 0

class HFDatasetTorch(Dataset):
    def __init__(self, hf_dataset, transform=None, indices=None):
        self.hf_dataset = hf_dataset
        self.transform = transform
        self.indices = indices if indices is not None else list(range(len(hf_dataset)))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        sample = self.hf_dataset[idx]
        img = sample["image"]
        label = int(sample["label"])

        if self.transform is not None:
            img = self.transform(img)

        return img, label, idx

## 2.4 Transform

In [ ]:
IMAGE_SIZE = 32
BATCH_SIZE = 64
NUM_WORKERS = safe_num_workers(2)

transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

## 2.5 Construir subset (20%)

In [ ]:
INDEX_COLUMN = "train_id_original"
VAL_FRACTION = 0.2

train_ids = load_ids_from_training_csv(TRAINING_CSV_PATH, index_column=INDEX_COLUMN)

rng = np.random.default_rng(SEED)
train_ids = np.array(train_ids)
rng.shuffle(train_ids)

num_val = int(len(train_ids) * VAL_FRACTION)
val_indices = train_ids[:num_val].tolist()
subset_train_indices = train_ids[num_val:].tolist()

subset_train_ds = HFDatasetTorch(train_hf, transform=transform, indices=subset_train_indices)
subset_val_ds   = HFDatasetTorch(train_hf, transform=transform, indices=val_indices)

full_train_ds   = HFDatasetTorch(train_hf, transform=transform)
full_test_ds    = HFDatasetTorch(test_hf, transform=transform)

print("Subset train dataset length:", len(subset_train_ds))
print("Subset val dataset length  :", len(subset_val_ds))
print("Full train dataset length  :", len(full_train_ds))
print("Full test dataset length   :", len(full_test_ds))

subset_train_loader = DataLoader(
    subset_train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

subset_val_loader = DataLoader(
    subset_val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

full_train_loader = DataLoader(
    full_train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

full_test_loader = DataLoader(
    full_test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("Subset train batches:", len(subset_train_loader))
print("Subset val batches  :", len(subset_val_loader))
print("Full train batches  :", len(full_train_loader))
print("Full test batches   :", len(full_test_loader))

# 3. Visualização dados

In [ ]:
def denorm(x):
    return x * 0.5 + 0.5

def show_batch_grid(loader, class_names, n_images=36, nrow=6, title="Sample Grid"):
    x, y, idx = next(iter(loader))
    x = x[:n_images]
    y = y[:n_images]

    grid = make_grid(denorm(x), nrow=nrow, padding=2)
    np_img = grid.permute(1, 2, 0).cpu().numpy()

    plt.figure(figsize=(8, 8))
    plt.imshow(np.clip(np_img, 0, 1))
    plt.axis("off")
    plt.title(title)
    plt.show()

    labels_str = [class_names[int(v)] for v in y]
    print("Labels:", labels_str)

show_batch_grid(subset_train_loader, class_names, n_images=36, nrow=6, title="ArtBench-10 Subset Train Samples")

In [ ]:
sample_img, sample_label, sample_idx = subset_train_ds[0]
print("Image shape:", sample_img.shape)
print("Label      :", sample_label, class_names[sample_label])
print("Index      :", sample_idx)
print("Min / Max  :", sample_img.min().item(), sample_img.max().item())

labels_train = [subset_train_ds[i][1] for i in range(len(subset_train_ds))]
labels_val   = [subset_val_ds[i][1] for i in range(len(subset_val_ds))]

class_counts_train = Counter(labels_train)
class_counts_val = Counter(labels_val)

print("\nDistribuição de classes (Subset Train):")
for cid, name in enumerate(class_names):
    print(f"{cid:2d} | {name:>20s} | {class_counts_train.get(cid, 0):6d}")

print("\nDistribuição de classes (Subset Val):")
for cid, name in enumerate(class_names):
    print(f"{cid:2d} | {name:>20s} | {class_counts_val.get(cid, 0):6d}")

# 5. Modelo DCGAN

## 5.1 Blocos auxiliares

In [ ]:
class PixelNorm(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x, eps=1e-8):
        return x / torch.sqrt(torch.mean(x**2, dim=1, keepdim=True) + eps)

def maybe_sn(layer, use_spectral_norm):
    if use_spectral_norm:
        return nn.utils.spectral_norm(layer)
    return layer

def weights_init(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        if hasattr(m, "weight") and m.weight is not None:
            nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        if hasattr(m, "weight") and m.weight is not None:
            nn.init.normal_(m.weight.data, 1.0, 0.02)
        if hasattr(m, "bias") and m.bias is not None:
            nn.init.constant_(m.bias.data, 0)

## 5.2 Gerador e Discriminador

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=128, g_features=64, use_pixelnorm=True):
        super().__init__()

        def g_block(in_ch, out_ch, k, s, p, use_pn=True):
            layers = [
                nn.ConvTranspose2d(in_ch, out_ch, k, s, p, bias=False),
                nn.BatchNorm2d(out_ch),
            ]
            if use_pn and use_pixelnorm:
                layers.append(PixelNorm())
            layers.append(nn.ReLU(True))
            return nn.Sequential(*layers)

        self.net = nn.Sequential(
            g_block(latent_dim, g_features * 8, 4, 1, 0),       # 4x4
            g_block(g_features * 8, g_features * 4, 4, 2, 1),   # 8x8
            g_block(g_features * 4, g_features * 2, 4, 2, 1),   # 16x16
            nn.ConvTranspose2d(g_features * 2, 3, 4, 2, 1, bias=False),  # 32x32
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, d_features=64, use_spectral_norm=True):
        super().__init__()

        self.net = nn.Sequential(
            maybe_sn(nn.Conv2d(3, d_features, 4, 2, 1, bias=False), use_spectral_norm),
            nn.LeakyReLU(0.2, inplace=True),

            maybe_sn(nn.Conv2d(d_features, d_features * 2, 4, 2, 1, bias=False), use_spectral_norm),
            nn.BatchNorm2d(d_features * 2),
            nn.LeakyReLU(0.2, inplace=True),

            maybe_sn(nn.Conv2d(d_features * 2, d_features * 4, 4, 2, 1, bias=False), use_spectral_norm),
            nn.BatchNorm2d(d_features * 4),
            nn.LeakyReLU(0.2, inplace=True),

            maybe_sn(nn.Conv2d(d_features * 4, 1, 4, 1, 0, bias=False), use_spectral_norm),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).view(-1)


@dataclass
class DCGANConfig:
    model_name: str = "dcgan"
    latent_dim: int = 128
    g_features: int = 64
    d_features: int = 64
    batch_size: int = 64
    epochs: int = 100
    lr_g: float = 2e-4
    lr_d: float = 1e-4
    beta1: float = 0.5
    beta2: float = 0.999
    label_smoothing: float = 0.1
    use_pixelnorm: bool = True
    use_spectral_norm: bool = True
    d_steps: int = 2
    g_steps: int = 1
    seed: int = 42
    save_every: int = 10


# 5.3 FID e KID

In [ ]:
def to_uint8_images(x):
    """
    Input: tensor normalized in [-1,1], shape [N,3,H,W]
    Output: uint8 in [0,255]
    """
    x = denorm(x).clamp(0, 1)
    x = (x * 255.0).round().to(torch.uint8)
    return x

metric_batch_size = 128

subset_val_metric_loader = DataLoader(
    subset_val_ds,
    batch_size=metric_batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

print("Val metric batches:", len(subset_val_metric_loader))


torch.no_grad()
def evaluate_generator_fid_kid(generator, real_loader, latent_dim, n_samples=1000, device=DEVICE):
    generator.eval()

    fid = FrechetInceptionDistance(feature=2048, normalize=False).to(device)
    kid = KernelInceptionDistance(subset_size=100, normalize=False).to(device)

    # real images
    real_count = 0
    for real_images, _, _ in real_loader:
        real_images = real_images.to(device)
        real_uint8 = to_uint8_images(real_images)

        needed = min(real_uint8.size(0), n_samples - real_count)
        if needed <= 0:
            break

        fid.update(real_uint8[:needed], real=True)
        kid.update(real_uint8[:needed], real=True)

        real_count += needed
        if real_count >= n_samples:
            break

    # fake images
    fake_count = 0
    batch_size = real_loader.batch_size if real_loader.batch_size is not None else 64

    while fake_count < n_samples:
        cur_bsz = min(batch_size, n_samples - fake_count)
        noise = torch.randn(cur_bsz, latent_dim, 1, 1, device=device)
        fake_images = generator(noise)
        fake_uint8 = to_uint8_images(fake_images)

        fid.update(fake_uint8, real=False)
        kid.update(fake_uint8, real=False)

        fake_count += cur_bsz

    fid_score = float(fid.compute().detach().cpu())
    kid_mean, kid_std = kid.compute()
    kid_mean = float(kid_mean.detach().cpu())
    kid_std = float(kid_std.detach().cpu())

    return {
        "fid": fid_score,
        "kid_mean": kid_mean,
        "kid_std": kid_std,
    }

@torch.no_grad()
def final_subset_evaluation(generator, latent_dim, real_loader, device=DEVICE):
    return evaluate_generator_fid_kid(
        generator=generator,
        real_loader=real_loader,
        latent_dim=latent_dim,
        n_samples=5000,
        device=device
    )

torch.no_grad()
def evaluate_generator_fid_kid(generator, real_loader, latent_dim, n_samples=1000, device=DEVICE):
    generator.eval()

    fid = FrechetInceptionDistance(feature=2048, normalize=False).to(device)
    kid = KernelInceptionDistance(subset_size=100, normalize=False).to(device)

    # real images
    real_count = 0
    for real_images, _, _ in real_loader:
        real_images = real_images.to(device)
        real_uint8 = to_uint8_images(real_images)

        needed = min(real_uint8.size(0), n_samples - real_count)
        if needed <= 0:
            break

        fid.update(real_uint8[:needed], real=True)
        kid.update(real_uint8[:needed], real=True)

        real_count += needed
        if real_count >= n_samples:
            break

    # fake images
    fake_count = 0
    batch_size = real_loader.batch_size if real_loader.batch_size is not None else 64

    while fake_count < n_samples:
        cur_bsz = min(batch_size, n_samples - fake_count)
        noise = torch.randn(cur_bsz, latent_dim, 1, 1, device=device)
        fake_images = generator(noise)
        fake_uint8 = to_uint8_images(fake_images)

        fid.update(fake_uint8, real=False)
        kid.update(fake_uint8, real=False)

        fake_count += cur_bsz

    fid_score = float(fid.compute().detach().cpu())
    kid_mean, kid_std = kid.compute()
    kid_mean = float(kid_mean.detach().cpu())
    kid_std = float(kid_std.detach().cpu())

    return {
        "fid": fid_score,
        "kid_mean": kid_mean,
        "kid_std": kid_std,
    }

@torch.no_grad()
def final_subset_evaluation(generator, latent_dim, real_loader, device=DEVICE):
    return evaluate_generator_fid_kid(
        generator=generator,
        real_loader=real_loader,
        latent_dim=latent_dim,
        n_samples=5000,
        device=device
    )


## 5.4 Early Stopping

In [ ]:
class EarlyStopper:
    def __init__(self, patience=4, min_delta=0.5):
        self.patience = patience
        self.min_delta = min_delta
        self.best_score = None
        self.bad_rounds = 0

    def step(self, score):
        # lower is better (FID)
        if self.best_score is None:
            self.best_score = score
            self.bad_rounds = 0
            return False, True

        if score < self.best_score - self.min_delta:
            self.best_score = score
            self.bad_rounds = 0
            return False, True

        self.bad_rounds += 1
        should_stop = self.bad_rounds >= self.patience
        return should_stop, False

# 5.5 Treino DCGAN

In [ ]:
def train_dcgan_with_early_stopping(
    cfg,
    train_loader,
    val_loader_metric,
    quick_eval_samples=1000,
    eval_every=5,
    patience=4,
    min_delta=0.5,
):
    seed_everything(cfg.seed)

    netG = Generator(
        latent_dim=cfg.latent_dim,
        g_features=cfg.g_features,
        use_pixelnorm=cfg.use_pixelnorm
    ).to(DEVICE)

    netD = Discriminator(
        d_features=cfg.d_features,
        use_spectral_norm=cfg.use_spectral_norm
    ).to(DEVICE)

    netG.apply(weights_init)
    netD.apply(weights_init)

    criterion = nn.BCELoss()

    optimizerD = optim.Adam(netD.parameters(), lr=cfg.lr_d, betas=(cfg.beta1, cfg.beta2))
    optimizerG = optim.Adam(netG.parameters(), lr=cfg.lr_g, betas=(cfg.beta1, cfg.beta2))

    run_name = (
        f"{cfg.model_name}"
        f"_ld{cfg.latent_dim}"
        f"_gf{cfg.g_features}"
        f"_df{cfg.d_features}"
        f"_bs{cfg.batch_size}"
        f"_lrg{cfg.lr_g}"
        f"_lrd{cfg.lr_d}"
        f"_b1{cfg.beta1}"
        f"_ls{cfg.label_smoothing}"
        f"_pn{int(cfg.use_pixelnorm)}"
        f"_sn{int(cfg.use_spectral_norm)}"
        f"_ds{cfg.d_steps}"
        f"_seed{cfg.seed}"
    )

    history = {
        "g_loss": [],
        "d_loss": [],
        "quick_fid": [],
        "quick_kid_mean": [],
        "quick_kid_std": [],
        "eval_epochs": [],
    }

    stopper = EarlyStopper(patience=patience, min_delta=min_delta)

    best_fid = float("inf")
    best_epoch = 0
    best_state_G = None
    best_state_D = None

    fixed_noise_local = torch.randn(64, cfg.latent_dim, 1, 1, device=DEVICE)

    print(f"\n===== TRAINING {run_name} =====")

    for epoch in range(1, cfg.epochs + 1):
        start_epoch = time.time()

        netG.train()
        netD.train()

        epoch_g_loss = 0.0
        epoch_d_loss = 0.0
        epoch_dx = 0.0
        epoch_dgz = 0.0

        for real_images, _, _ in tqdm(train_loader, leave=False, desc=f"Epoch {epoch}/{cfg.epochs}"):
            real_images = real_images.to(DEVICE)
            bsz = real_images.size(0)

            d_running = 0.0
            dx_running = 0.0
            dgz_running = 0.0

            for _ in range(cfg.d_steps):
                optimizerD.zero_grad()

                real_labels = torch.full((bsz,), 1.0 - cfg.label_smoothing, device=DEVICE)
                fake_labels = torch.zeros(bsz, device=DEVICE)

                output_real = netD(real_images)
                loss_real = criterion(output_real, real_labels)

                noise = torch.randn(bsz, cfg.latent_dim, 1, 1, device=DEVICE)
                fake_images = netG(noise)

                output_fake = netD(fake_images.detach())
                loss_fake = criterion(output_fake, fake_labels)

                loss_D = loss_real + loss_fake
                loss_D.backward()
                optimizerD.step()

                d_running += loss_D.item()
                dx_running += output_real.mean().item()
                dgz_running += output_fake.mean().item()

            g_running = 0.0
            for _ in range(cfg.g_steps):
                optimizerG.zero_grad()

                noise = torch.randn(bsz, cfg.latent_dim, 1, 1, device=DEVICE)
                fake_images = netG(noise)

                target_gen = torch.ones(bsz, device=DEVICE)
                output_fake_for_g = netD(fake_images)
                loss_G = criterion(output_fake_for_g, target_gen)

                loss_G.backward()
                optimizerG.step()

                g_running += loss_G.item()

            epoch_d_loss += d_running / cfg.d_steps
            epoch_g_loss += g_running / cfg.g_steps
            epoch_dx += dx_running / cfg.d_steps
            epoch_dgz += dgz_running / cfg.d_steps

        epoch_d_loss /= len(train_loader)
        epoch_g_loss /= len(train_loader)
        epoch_dx /= len(train_loader)
        epoch_dgz /= len(train_loader)

        history["d_loss"].append(epoch_d_loss)
        history["g_loss"].append(epoch_g_loss)

        elapsed = time.time() - start_epoch
        print(
            f"Epoch [{epoch:03d}/{cfg.epochs}] | "
            f"D Loss: {epoch_d_loss:.4f} | G Loss: {epoch_g_loss:.4f} | "
            f"D(x): {epoch_dx:.4f} | D(G(z)): {epoch_dgz:.4f} | Time: {elapsed:.1f}s"
        )

        if epoch % cfg.save_every == 0 or epoch == 1 or epoch == cfg.epochs:
            netG.eval()
            with torch.no_grad():
                fake_fixed = netG(fixed_noise_local)
            save_generated_grid(fake_fixed, epoch, run_name, out_dir=IMAGES_DIR, show=False)

        if epoch % eval_every == 0 or epoch == cfg.epochs:
            quick_metrics = evaluate_generator_fid_kid(
                generator=netG,
                real_loader=val_loader_metric,
                latent_dim=cfg.latent_dim,
                n_samples=quick_eval_samples,
                device=DEVICE
            )

            qfid = quick_metrics["fid"]
            qkid = quick_metrics["kid_mean"]
            qkid_std = quick_metrics["kid_std"]

            history["quick_fid"].append(qfid)
            history["quick_kid_mean"].append(qkid)
            history["quick_kid_std"].append(qkid_std)
            history["eval_epochs"].append(epoch)

            print(
                f"  -> Quick Eval @ epoch {epoch}: "
                f"FID={qfid:.3f}, KID={qkid:.5f} ± {qkid_std:.5f}"
            )

            should_stop, improved = stopper.step(qfid)

            if improved:
                best_fid = qfid
                best_epoch = epoch
                best_state_G = copy.deepcopy(netG.state_dict())
                best_state_D = copy.deepcopy(netD.state_dict())

            if should_stop:
                print(f"  -> Early stopping triggered at epoch {epoch}")
                break

    if best_state_G is None:
        best_state_G = copy.deepcopy(netG.state_dict())
        best_state_D = copy.deepcopy(netD.state_dict())
        best_epoch = cfg.epochs

    netG.load_state_dict(best_state_G)
    netD.load_state_dict(best_state_D)

    model_path_G = MODELS_DIR / f"{run_name}_BEST_G.pt"
    model_path_D = MODELS_DIR / f"{run_name}_BEST_D.pt"

    torch.save(netG.state_dict(), model_path_G)
    torch.save(netD.state_dict(), model_path_D)

    print(f"Best epoch: {best_epoch} | Best quick FID: {best_fid:.3f}")
    print("Saved generator:", model_path_G)
    print("Saved discriminator:", model_path_D)

    return netG, netD, history, run_name, best_epoch, best_fid

## 5.6 Grid Search

In [ ]:
grid_space = {
    "latent_dim": [96, 128],
    "g_features": [64],
    "d_features": [64],
    "batch_size": [64],
    "epochs": [100],
    "lr_g": [2e-4, 3e-4],
    "lr_d": [1e-4],
    "beta1": [0.5],
    "beta2": [0.999],
    "label_smoothing": [0.05, 0.10],
    "use_pixelnorm": [True],
    "use_spectral_norm": [True],
    "d_steps": [1, 2],
    "g_steps": [1],
    "save_every": [10],
}

def build_grid_configs(grid_space, seed=42, model_name="dcgan_grid_subset"):
    keys = list(grid_space.keys())
    values = list(grid_space.values())
    configs = []

    for combo in product(*values):
        kwargs = dict(zip(keys, combo))
        cfg = DCGANConfig(
            model_name=model_name,
            seed=seed,
            **kwargs
        )
        configs.append(cfg)

    return configs

grid_configs = build_grid_configs(grid_space, seed=42)
print("Número de configs da grid:", len(grid_configs))

## 5.7 Run Grid Search (subset)

In [ ]:
grid_results = []

for cfg in grid_configs:
    print("\n" + "=" * 100)
    print("GRID RUN")
    print(cfg)
    print("=" * 100)

    train_loader = DataLoader(
        subset_train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    netG, netD, history, run_name, best_epoch, best_quick_fid = train_dcgan_with_early_stopping(
        cfg=cfg,
        train_loader=train_loader,
        val_loader_metric=subset_val_metric_loader,
        quick_eval_samples=1000,
        eval_every=5,
        patience=4,
        min_delta=0.5,
    )

    final_metrics = final_subset_evaluation(
        generator=netG,
        latent_dim=cfg.latent_dim,
        real_loader=subset_val_metric_loader,
        device=DEVICE
    )

    row = asdict(cfg)
    row.update({
        "run_name": run_name,
        "best_epoch": best_epoch,
        "best_quick_fid": best_quick_fid,
        "final_subset_fid": final_metrics["fid"],
        "final_subset_kid_mean": final_metrics["kid_mean"],
        "final_subset_kid_std": final_metrics["kid_std"],
    })
    grid_results.append(row)

    history_path = METRICS_DIR / f"{run_name}_history.json"
    with open(history_path, "w") as f:
        json.dump(history, f, indent=2)

    del netG, netD
    torch.cuda.empty_cache()

grid_df = pd.DataFrame(grid_results)
grid_csv_path = METRICS_DIR / "dcgan_grid_subset_results.csv"
grid_df.to_csv(grid_csv_path, index=False)

print("Saved:", grid_csv_path)
print(grid_df.head())

## 5.8 Melhores 3 Configuraçoes

In [ ]:
grid_summary = (
    grid_df
    .sort_values(by=["final_subset_fid", "final_subset_kid_mean"], ascending=[True, True])
    .reset_index(drop=True)
)

grid_summary_path = METRICS_DIR / "dcgan_grid_subset_ranking.csv"
grid_summary.to_csv(grid_summary_path, index=False)

print("Saved:", grid_summary_path)
print(grid_summary.head())

In [ ]:
top_k = 3
top_grid_df = grid_summary.head(top_k).copy()
print(top_grid_df)

def row_to_dcgan_config(row):
    return DCGANConfig(
        model_name=row["model_name"],
        latent_dim=int(row["latent_dim"]),
        g_features=int(row["g_features"]),
        d_features=int(row["d_features"]),
        batch_size=int(row["batch_size"]),
        epochs=int(row["epochs"]),
        lr_g=float(row["lr_g"]),
        lr_d=float(row["lr_d"]),
        beta1=float(row["beta1"]),
        beta2=float(row["beta2"]),
        label_smoothing=float(row["label_smoothing"]),
        use_pixelnorm=bool(row["use_pixelnorm"]),
        use_spectral_norm=bool(row["use_spectral_norm"]),
        d_steps=int(row["d_steps"]),
        g_steps=int(row["g_steps"]),
        seed=42,
        save_every=int(row["save_every"]),
    )

top_configs = [row_to_dcgan_config(row) for _, row in top_grid_df.iterrows()]

## 5.10 Run com as 10 seeds para as 3 melhores combinações

In [ ]:
selection_seeds = [42, 123, 999, 2024, 7, 11, 21, 33, 77, 101]

top_results = []

for base_cfg in top_configs:
    for seed in selection_seeds:
        cfg = copy.deepcopy(base_cfg)
        cfg.seed = seed
        cfg.model_name = "dcgan_top10seeds_subset"

        print("\n" + "=" * 100)
        print("TOP CONFIG × SEED")
        print(cfg)
        print("=" * 100)

        train_loader = DataLoader(
            subset_train_ds,
            batch_size=cfg.batch_size,
            shuffle=True,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
        )

        netG, netD, history, run_name, best_epoch, best_quick_fid = train_dcgan_with_early_stopping(
            cfg=cfg,
            train_loader=train_loader,
            val_loader_metric=subset_val_metric_loader,
            quick_eval_samples=1000,
            eval_every=5,
            patience=4,
            min_delta=0.5,
        )

        final_metrics = final_subset_evaluation(
            generator=netG,
            latent_dim=cfg.latent_dim,
            real_loader=subset_val_metric_loader,
            device=DEVICE
        )

        row = asdict(cfg)
        row.update({
            "run_name": run_name,
            "best_epoch": best_epoch,
            "best_quick_fid": best_quick_fid,
            "final_subset_fid": final_metrics["fid"],
            "final_subset_kid_mean": final_metrics["kid_mean"],
            "final_subset_kid_std": final_metrics["kid_std"],
        })
        top_results.append(row)

        history_path = METRICS_DIR / f"{run_name}_history.json"
        with open(history_path, "w") as f:
            json.dump(history, f, indent=2)

        del netG, netD
        torch.cuda.empty_cache()

top_results_df = pd.DataFrame(top_results)
top_results_csv = METRICS_DIR / "dcgan_topconfigs_10seeds_subset.csv"
top_results_df.to_csv(top_results_csv, index=False)

print("Saved:", top_results_csv)
print(top_results_df.head())

In [ ]:
metrics_dir = Path("/content/drive/MyDrive/IAG/results/metrics")

csv_path = metrics_dir / "dcgan_topconfigs_10seeds_subset.csv"
df = pd.read_csv(csv_path)

group_cols = [
    "latent_dim", "g_features", "d_features", "batch_size",
    "lr_g", "lr_d", "beta1", "beta2", "label_smoothing",
    "use_pixelnorm", "use_spectral_norm", "d_steps", "g_steps"
]

summary = (
    df
    .groupby(group_cols, dropna=False)
    .agg(
        fid_mean=("final_subset_fid", "mean"),
        fid_std=("final_subset_fid", "std"),
        kid_mean=("final_subset_kid_mean", "mean"),
        kid_std=("final_subset_kid_mean", "std"),
        best_epoch_mean=("best_epoch", "mean"),
    )
    .reset_index()
    .sort_values("fid_mean", ascending=True)
)

summary

## 5.9 Visualizacao Imagens

In [ ]:
class PixelNorm(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x, eps=1e-8):
        return x / torch.sqrt(torch.mean(x**2, dim=1, keepdim=True) + eps)

class Generator(nn.Module):
    def __init__(self, latent_dim=128, g_features=64, use_pixelnorm=True):
        super().__init__()

        def g_block(in_ch, out_ch, k, s, p, use_pn=True):
            layers = [
                nn.ConvTranspose2d(in_ch, out_ch, k, s, p, bias=False),
                nn.BatchNorm2d(out_ch),
            ]
            if use_pn and use_pixelnorm:
                layers.append(PixelNorm())
            layers.append(nn.ReLU(True))
            return nn.Sequential(*layers)

        self.net = nn.Sequential(
            g_block(latent_dim, g_features * 8, 4, 1, 0),       # 4x4
            g_block(g_features * 8, g_features * 4, 4, 2, 1),   # 8x8
            g_block(g_features * 4, g_features * 2, 4, 2, 1),   # 16x16
            nn.ConvTranspose2d(g_features * 2, 3, 4, 2, 1, bias=False),  # 32x32
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)

metrics_dir = Path("/content/drive/MyDrive/IAG/results/metrics")
models_dir = Path("/content/drive/MyDrive/IAG/results/models")

csv_path = metrics_dir / "dcgan_topconfigs_10seeds_subset.csv"
df = pd.read_csv(csv_path)

top3 = df.sort_values("final_subset_fid").head(3)
display(top3[["run_name", "final_subset_fid", "latent_dim", "g_features", "use_pixelnorm"]])

In [ ]:
fixed_z = {}

for _, row in top3.iterrows():
    ld = int(row["latent_dim"])
    if ld not in fixed_z:
        fixed_z[ld] = torch.randn(64, ld, 1, 1, device=DEVICE)

for _, row in top3.iterrows():
    run_name = row["run_name"]
    model_path = models_dir / f"{run_name}_BEST_G.pt"

    latent_dim = int(row["latent_dim"])
    g_features = int(row["g_features"])
    use_pixelnorm = bool(row["use_pixelnorm"])

    print("\nRUN:", run_name)
    print("FID:", row["final_subset_fid"])
    print("Model exists:", model_path.exists())

    G = Generator(
        latent_dim=latent_dim,
        g_features=g_features,
        use_pixelnorm=use_pixelnorm
    ).to(DEVICE)

    state = torch.load(model_path, map_location=DEVICE)
    G.load_state_dict(state)
    G.eval()

    with torch.no_grad():
        fake = G(fixed_z[latent_dim]).cpu()

    grid = make_grid(fake, nrow=8, normalize=True, value_range=(-1, 1))

    plt.figure(figsize=(7, 7))
    plt.imshow(grid.permute(1, 2, 0))
    plt.title(f"{run_name}\nFID={row['final_subset_fid']:.2f}")
    plt.axis("off")
    plt.show()

## 6. Run dataset total

In [ ]:
best_full_cfg = DCGANConfig(
    model_name="dcgan_full_final",
    latent_dim=96,
    g_features=64,
    d_features=64,
    batch_size=64,
    epochs=100,
    lr_g=3e-4,
    lr_d=1e-4,
    beta1=0.5,
    beta2=0.999,
    label_smoothing=0.10,
    use_pixelnorm=True,
    use_spectral_norm=True,
    d_steps=2,
    g_steps=1,
    seed=42,
    save_every=10,
)

final_seeds = [42, 123, 999, 2024, 7, 11, 21, 33, 77, 101]

full_train_loader = DataLoader(
    full_train_ds,
    batch_size=best_full_cfg.batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

full_test_metric_loader = DataLoader(
    full_test_ds,
    batch_size=128,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

full_results_csv = METRICS_DIR / "dcgan_full_final_10seeds.csv"
full_summary_csv = METRICS_DIR / "dcgan_full_final_10seeds_summary.csv"
completed_seeds_json = METRICS_DIR / "dcgan_full_final_completed_seeds.json"


def load_completed_seeds(json_path):
    if json_path.exists():
        with open(json_path, "r") as f:
            return set(json.load(f))
    return set()

def save_completed_seeds(json_path, completed_seeds):
    with open(json_path, "w") as f:
        json.dump(sorted(list(completed_seeds)), f, indent=2)

def load_existing_results(csv_path):
    if csv_path.exists():
        return pd.read_csv(csv_path)
    return pd.DataFrame()

def save_generated_grid(fake_images, epoch, run_name, out_dir, nrow=8, show=False):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # fake_images já vem como tensor [-1, 1]
    fake_images = ((fake_images.detach().cpu() + 1) / 2).clamp(0, 1)

    grid = make_grid(fake_images, nrow=nrow, padding=2)
    np_img = grid.permute(1, 2, 0).numpy()

    plt.figure(figsize=(8, 8))
    plt.imshow(np.clip(np_img, 0, 1))
    plt.axis("off")
    plt.title(f"{run_name} - epoch {epoch}")

    save_path = out_dir / f"{run_name}_epoch_{epoch:03d}.png"
    plt.savefig(save_path, bbox_inches="tight")
    if show:
        plt.show()
    plt.close()

    return save_path

def denorm(x):
    return ((x + 1) / 2).clamp(0, 1)

In [ ]:
completed_seeds = load_completed_seeds(completed_seeds_json)
existing_df = load_existing_results(full_results_csv)

if not existing_df.empty:
    print(f"Já existem {len(existing_df)} resultados guardados.")
else:
    print("Ainda não existem resultados guardados.")

print("Seeds já concluídos:", completed_seeds)

for seed in final_seeds:
    if seed in completed_seeds:
        print(f"\n[SKIP] Seed {seed} já concluída.")
        continue

    cfg = copy.deepcopy(best_full_cfg)
    cfg.seed = seed

    print("\n" + "=" * 100)
    print(f"FULL DATASET RUN | seed={seed}")
    print(cfg)
    print("=" * 100)

    try:
        netG, netD, history, run_name, best_epoch, best_quick_fid = train_dcgan_with_early_stopping(
            cfg=cfg,
            train_loader=full_train_loader,
            val_loader_metric=full_test_metric_loader,
            quick_eval_samples=1000,
            eval_every=5,
            patience=4,
            min_delta=0.5,
        )

        final_metrics = final_subset_evaluation(
            generator=netG,
            latent_dim=cfg.latent_dim,
            real_loader=full_test_metric_loader,
            device=DEVICE
        )

        row = asdict(cfg)
        row.update({
            "run_name": run_name,
            "best_epoch": best_epoch,
            "best_quick_fid": best_quick_fid,
            "final_full_fid": final_metrics["fid"],
            "final_full_kid_mean": final_metrics["kid_mean"],
            "final_full_kid_std": final_metrics["kid_std"],
        })

        # acrescentar ao dataframe existente
        if existing_df.empty:
            existing_df = pd.DataFrame([row])
        else:
            existing_df = pd.concat([existing_df, pd.DataFrame([row])], ignore_index=True)

        # guardar histórico da seed
        history_path = METRICS_DIR / f"{run_name}_history.json"
        with open(history_path, "w") as f:
            json.dump(history, f, indent=2)

        # guardar CSV logo após cada seed
        existing_df.to_csv(full_results_csv, index=False)

        # marcar seed como concluída
        completed_seeds.add(seed)
        save_completed_seeds(completed_seeds_json, completed_seeds)

        print(f"[OK] Seed {seed} concluída e guardada.")
        print(f"Current results saved to: {full_results_csv}")

        del netG, netD
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"[ERROR] Seed {seed} falhou: {e}")
        torch.cuda.empty_cache()
        break

In [ ]:
full_results_df = pd.read_csv(full_results_csv)
full_results_df

In [ ]:
full_results_df = pd.read_csv(full_results_csv)

full_summary_df = pd.DataFrame([{
    "latent_dim": full_results_df["latent_dim"].iloc[0],
    "g_features": full_results_df["g_features"].iloc[0],
    "d_features": full_results_df["d_features"].iloc[0],
    "batch_size": full_results_df["batch_size"].iloc[0],
    "lr_g": full_results_df["lr_g"].iloc[0],
    "lr_d": full_results_df["lr_d"].iloc[0],
    "beta1": full_results_df["beta1"].iloc[0],
    "beta2": full_results_df["beta2"].iloc[0],
    "label_smoothing": full_results_df["label_smoothing"].iloc[0],
    "use_pixelnorm": full_results_df["use_pixelnorm"].iloc[0],
    "use_spectral_norm": full_results_df["use_spectral_norm"].iloc[0],
    "d_steps": full_results_df["d_steps"].iloc[0],
    "g_steps": full_results_df["g_steps"].iloc[0],
    "fid_mean": full_results_df["final_full_fid"].mean(),
    "fid_std": full_results_df["final_full_fid"].std(),
    "kid_mean": full_results_df["final_full_kid_mean"].mean(),
    "kid_std": full_results_df["final_full_kid_mean"].std(),
    "best_epoch_mean": full_results_df["best_epoch"].mean(),
    "num_completed_seeds": len(full_results_df),
}])

full_summary_df.to_csv(full_summary_csv, index=False)

print("Saved:", full_summary_csv)
display(full_summary_df)

## 6.1 Visualizar Imagens

In [ ]:
METRICS_DIR = Path("/content/drive/MyDrive/IAG/results/metrics")
MODELS_DIR = Path("/content/drive/MyDrive/IAG/results/models")

full_results_csv = METRICS_DIR / "dcgan_full_final_10seeds.csv"
full_results_df = pd.read_csv(full_results_csv)

best_row = full_results_df.sort_values("final_full_fid").iloc[0]

best_model_path = MODELS_DIR / f"{best_row['run_name']}_BEST_G.pt"
print(best_model_path)
print("Existe?", best_model_path.exists())

netG = Generator(
    latent_dim=int(best_row["latent_dim"]),
    g_features=int(best_row["g_features"]),
    use_pixelnorm=bool(best_row["use_pixelnorm"])
).to(DEVICE)

state_dict = torch.load(best_model_path, map_location=DEVICE)
netG.load_state_dict(state_dict)
netG.eval()


def denorm(x):
    return ((x + 1) / 2).clamp(0, 1)

def get_random_real_images(dataset, n=10):
    indices = torch.randperm(len(dataset))[:n]
    imgs = [dataset[i][0] for i in indices]
    return torch.stack(imgs)

def generate_fake_images(generator, latent_dim, n, device):
    generator.eval()
    with torch.no_grad():
        z = torch.randn(n, latent_dim, 1, 1, device=device)
        fake = generator(z)
    return denorm(fake.cpu())

def plot_block(real_imgs, fake_imgs, block_id):
    n = real_imgs.shape[0]
    fig, axes = plt.subplots(2, n, figsize=(20, 4))

    for i in range(n):
        axes[0, i].imshow(real_imgs[i].permute(1, 2, 0))
        axes[0, i].axis("off")

    for i in range(n):
        axes[1, i].imshow(fake_imgs[i].permute(1, 2, 0))
        axes[1, i].axis("off")

    axes[0, 0].set_ylabel("Real", fontsize=12)
    axes[1, 0].set_ylabel("Generated", fontsize=12)

    plt.suptitle(f"Block {block_id}", fontsize=14)
    plt.tight_layout()
    plt.show()



In [ ]:
latent_dim_best = int(best_row["latent_dim"])

for block in range(4):
    real_imgs = get_random_real_images(full_train_ds, n=10)
    real_imgs = denorm(real_imgs)

    fake_imgs = generate_fake_images(
        generator=netG,
        latent_dim=latent_dim_best,
        n=10,
        device=DEVICE
    )

    plot_block(real_imgs, fake_imgs, block + 1)